<a href="https://colab.research.google.com/github/wspiva-wingulogic/advance-your-sql-skills-for-data-engineering-3315012/blob/main/6_17_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# PANDAS DEMO: Real-World Data Cleaning Investigation (100k rows)
# ============================================================
# You just received 100k rows of finance transaction data from
# upstream. You've never seen this dataset before. You can't
# eyeball 100k rows. Your job: figure out what's wrong, then
# fix it — using ONLY pandas diagnostics.
#
# Structure of every real cleaning job:
#   PART A — Receive the dataset                (Step 1)
#   PART B — Discovery: find what's wrong       (Steps 2–5)
#   PART C — Clean: address every issue found   (Step 6)
#   PART D — Verify: re-run diagnostics         (Step 7)
# ============================================================


# ------------------------------------------------------------
# STEP 1: Simulate a realistic messy transaction dataset
# ------------------------------------------------------------
# We build a clean base of 100k transactions, then INJECT the
# kinds of problems real finance data actually has: sign errors,
# duplicate rows, typos, whitespace, mixed case, missing IDs,
# data-entry outliers, future-dated rows.
#
# The issues are randomly SCATTERED through the dataset so you
# can't spot them by eye — only by diagnostics. In real life,
# this is just "the data we got from upstream."

import pandas as pd
import numpy as np

np.random.seed(42)
n = 100_000

df = pd.DataFrame({
    "transaction_id":   [f"TXN{i:07d}" for i in range(n)],
    "timestamp":        pd.date_range("2024-01-01", periods=n, freq="5min"),
    "customer_id":      np.random.randint(1000, 9999, size=n).astype(float),
    "amount":           np.round(np.random.lognormal(mean=4, sigma=1, size=n), 2),
    "currency":         "USD",
    "transaction_type": np.random.choice(["deposit", "withdrawal", "transfer", "fee"], size=n),
    "status":           "completed",
    "merchant":         np.random.choice(["amazon", "starbucks", "walmart", "uber", "spotify"], size=n),
})

# (1) Duplicate rows  (~2% of dataset)
dup_idx = np.random.choice(n, size=2000, replace=False)
df = pd.concat([df, df.iloc[dup_idx]], ignore_index=True)

# (2) Negative amounts on some deposits  (sign errors in source system)
neg_idx = df[df["transaction_type"] == "deposit"].sample(50, random_state=1).index
df.loc[neg_idx, "amount"] = -df.loc[neg_idx, "amount"]

# (3) Massive outliers  (data entry — extra zero or two)
out_idx = np.random.choice(len(df), size=30, replace=False)
df.loc[out_idx, "amount"] = df.loc[out_idx, "amount"] * 1000

# (4) Missing customer_ids  (system glitch)
miss_idx = np.random.choice(len(df), size=200, replace=False)
df.loc[miss_idx, "customer_id"] = np.nan

# (5) Inconsistent status values  (different upstream systems used different words)
incons_idx = np.random.choice(len(df), size=300, replace=False)
df.loc[incons_idx[:100],   "status"] = "complete"
df.loc[incons_idx[100:200], "status"] = "Success"
df.loc[incons_idx[200:],   "status"] = "OK"

# (6) Whitespace + case inconsistencies in string columns
ws_idx = np.random.choice(len(df), size=500, replace=False)
df.loc[ws_idx, "merchant"] = " " + df.loc[ws_idx, "merchant"] + " "
case_idx = np.random.choice(len(df), size=400, replace=False)
df.loc[case_idx, "transaction_type"] = df.loc[case_idx, "transaction_type"].str.upper()

# (7) A few non-USD transactions  (FX conversion not applied upstream)
fx_idx = np.random.choice(len(df), size=20, replace=False)
df.loc[fx_idx, "currency"] = np.random.choice(["EUR", "GBP"], size=20)

# (8) Typo in transaction_type
typo_idx = np.random.choice(len(df), size=10, replace=False)
df.loc[typo_idx, "transaction_type"] = "depost"

# (9) Future-dated transactions  (year typo)
future_idx = np.random.choice(len(df), size=5, replace=False)
df.loc[future_idx, "timestamp"] = pd.Timestamp("2099-01-01")

# Shuffle so issues are scattered, not clustered at the end
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"received: {len(df):,} rows from upstream\n")



received: 102,000 rows from upstream



In [2]:
# ============================================================
# PART B — DISCOVERY
# Pretend you've never seen this data. Use diagnostics, not eyes.
# ============================================================


# ------------------------------------------------------------
# STEP 2: First look — shape, types, summary stats
# ------------------------------------------------------------
# These 5 commands should be the FIRST thing you do on any new
# dataset, every time. We're not fixing anything yet — we're
# building a mental picture and spotting red flags.

print("--- head ---")
print(df.head())

print("\n--- shape ---")
print(df.shape)
# >>> (102000, 8) — we expected 100,000. RED FLAG: extra 2,000 rows
# means there are probably duplicates somewhere.

print("\n--- dtypes ---")
print(df.dtypes)
# customer_id is FLOAT64. Customer IDs should be ints — float on
# an integer column almost always means NaN values exist
# (pandas can't have NaN in an int column).

print("\n--- info ---")
df.info()

print("\n--- describe ---")
print(df.describe())
# amount: min is NEGATIVE — but most transaction types shouldn't allow that
# amount: max is HUGE compared to the 75th percentile — outliers
# timestamp max is in 2099 — future-dated rows exist
# customer_id count < total rows — confirms missing values



--- head ---
  transaction_id           timestamp  customer_id  amount currency  \
0     TXN0097413 2024-12-04 05:45:00       2259.0  233.03      USD   
1     TXN0009138 2024-02-01 17:30:00       6932.0   18.31      USD   
2     TXN0089078 2024-11-05 07:10:00       6076.0   48.45      USD   
3     TXN0042332 2024-05-26 23:40:00       1879.0   91.47      USD   
4     TXN0036832 2024-05-07 21:20:00       8526.0  102.51      USD   

  transaction_type     status   merchant  
0         transfer  completed  starbucks  
1       withdrawal  completed     amazon  
2         transfer  completed  starbucks  
3          deposit  completed       uber  
4          deposit  completed    spotify  

--- shape ---
(102000, 8)

--- dtypes ---
transaction_id              object
timestamp           datetime64[ns]
customer_id                float64
amount                     float64
currency                    object
transaction_type            object
status                      object
merchant            

In [4]:
# ------------------------------------------------------------
# STEP 3: Where are the missing values?
# ------------------------------------------------------------
# .isna().sum() = NaN count per column. Always run this on new data.

print("\n--- missing values ---")
print(df.isna().sum())
# Confirms: customer_id has ~200 NaN. Nothing else missing.



--- missing values ---
transaction_id        0
timestamp             0
customer_id         200
amount                0
currency              0
transaction_type      0
status                0
merchant              0
dtype: int64


In [ ]:
# ------------------------------------------------------------
# STEP 4: Categorical deep-dive with .value_counts()
# ------------------------------------------------------------
# value_counts() is the most underrated cleaning tool. It instantly
# surfaces typos, case mismatches, whitespace, and surprise values
# in any string column. Run it on EVERY object/string column.

print("\n--- currency ---")
print(df["currency"].value_counts())
# Mostly USD, but a few EUR and GBP rows — FX not converted

print("\n--- transaction_type ---")
print(df["transaction_type"].value_counts())
# "deposit" AND "DEPOSIT" both present — case inconsistency
# "depost" present — a typo
# (Same for withdrawal/transfer/fee in mixed case)

print("\n--- status ---")
print(df["status"].value_counts())
# "completed", "complete", "Success", "OK" — four ways of saying the same thing

print("\n--- merchant (top 15) ---")
print(df["merchant"].value_counts().head(15))
# Notice: "amazon" appears AND " amazon " (with whitespace) appears separately.
# These will look identical when printed but are different strings.




--- currency ---
currency
USD    101980
GBP        12
EUR         8
Name: count, dtype: int64

--- transaction_type ---
transaction_type
transfer      25489
deposit       25478
withdrawal    25358
fee           25265
WITHDRAWAL      116
DEPOSIT         103
FEE              91
TRANSFER         90
depost           10
Name: count, dtype: int64

--- status ---
status
completed    101700
OK              100
complete        100
Success         100
Name: count, dtype: int64

--- merchant (top 15) ---
merchant
spotify        20532
walmart        20352
uber           20341
starbucks      20231
amazon         20044
 starbucks       109
 amazon          108
 walmart         104
 spotify          93
 uber             86
Name: count, dtype: int64


In [ ]:
# ------------------------------------------------------------
# STEP 5: Sanity checks on numeric / date columns
# ------------------------------------------------------------
# .describe() raised flags. Now zoom in on each.

# Future-dated rows?
print("\n--- future-dated rows ---")
print(df[df["timestamp"] > pd.Timestamp("2025-12-31")].shape)

# Negative amounts? On what transaction types?
print("\n--- negative-amount rows by type ---")
neg = df[df["amount"] < 0]
print(neg.shape)
print(neg["transaction_type"].str.lower().str.strip().value_counts())

# How extreme are the outliers? Compare 99.9th percentile to max.
print("\n--- amount outliers ---")
print(f"99.9th percentile: {df['amount'].quantile(0.999):,.2f}")
print(f"max:               {df['amount'].max():,.2f}")
# If max >> 99.9th percentile, you have outliers.

# Duplicate transaction IDs? (IDs should be unique)
print("\n--- duplicate IDs ---")
print(df["transaction_id"].duplicated().sum())
# Confirms the 2,000 extra rows: duplicate transaction IDs.



--- future-dated rows ---
(5, 8)

--- negative-amount rows by type ---
(50, 8)
transaction_type
deposit    50
Name: count, dtype: int64

--- amount outliers ---
99.9th percentile: 1,276.34
max:               273,670.00

--- duplicate IDs ---
2000


In [6]:
# ============================================================
# PART C — CLEAN
# We found:
#   - duplicate transaction IDs
#   - customer_id NaNs (200)
#   - amount: negative deposits + huge outliers
#   - currency: a few non-USD rows
#   - transaction_type: case mix + typo "depost"
#   - status: 4 synonyms for "completed"
#   - merchant: whitespace duplicates
#   - timestamp: future-dated rows
# Fix in dependency order: standardize strings -> filter / fix
# values -> drop duplicates LAST.
# ============================================================


# ------------------------------------------------------------
# STEP 6: Apply each fix
# ------------------------------------------------------------

# (a) Standardize string columns: strip + lowercase
for col in ["transaction_type", "status", "merchant", "currency"]:
    df[col] = df[col].str.strip().str.lower()

# (b) Replace typos and synonyms via .replace with a dict
df["transaction_type"] = df["transaction_type"].replace({"depost": "deposit"})
df["status"] = df["status"].replace({
    "complete": "completed",
    "success":  "completed",
    "ok":       "completed",
})

# (c) Currency: filter to USD only (simplest move — could also FX-convert)
df = df[df["currency"] == "usd"].copy()

# (d) Missing customer_id: drop those rows, then convert back to int
df = df.dropna(subset=["customer_id"])
df["customer_id"] = df["customer_id"].astype(int)

# (e) Negative deposits: assume sign error, take absolute value
deposits = df["transaction_type"] == "deposit"
df.loc[deposits, "amount"] = df.loc[deposits, "amount"].abs()

# (f) Amount outliers: clip to the 99.9th percentile
cap = df["amount"].quantile(0.999)
df["amount"] = df["amount"].clip(upper=cap)

# (g) Future-dated rows: drop
df = df[df["timestamp"] <= pd.Timestamp("2025-12-31")]

# (h) Duplicate transaction IDs: drop, keep the first occurrence
df = df.drop_duplicates(subset=["transaction_id"], keep="first")

print(f"\nafter cleaning: {len(df):,} rows")




after cleaning: 99,783 rows


In [7]:
# ------------------------------------------------------------
# STEP 7: Verify — re-run the SAME diagnostics on the clean data
# ------------------------------------------------------------
# Always re-inspect after cleaning. If something snuck through,
# you'll catch it here — not in production.

print("\n--- shape ---")
print(df.shape)                                # back near 100k

print("\n--- dtypes ---")
print(df.dtypes)                               # customer_id back to int

print("\n--- missing ---")
print(df.isna().sum())                         # all zeros

print("\n--- transaction_type ---")
print(df["transaction_type"].value_counts())   # exactly 4 categories

print("\n--- status ---")
print(df["status"].value_counts())             # one value: "completed"

print("\n--- amount ---")
print(df["amount"].describe())                 # no negatives, max capped



--- shape ---
(99783, 8)

--- dtypes ---
transaction_id              object
timestamp           datetime64[ns]
customer_id                  int64
amount                     float64
currency                    object
transaction_type            object
status                      object
merchant                    object
dtype: object

--- missing ---
transaction_id      0
timestamp           0
customer_id         0
amount              0
currency            0
transaction_type    0
status              0
merchant            0
dtype: int64

--- transaction_type ---
transaction_type
transfer      25039
deposit       25034
withdrawal    24927
fee           24783
Name: count, dtype: int64

--- status ---
status
completed    99783
Name: count, dtype: int64

--- amount ---
count    99783.000000
mean        90.186953
std        113.628997
min          0.630000
25%         27.885000
50%         54.640000
75%        107.360000
max       1276.525967
Name: amount, dtype: float64


In [9]:
# a = df["timestamp"] <= pd.Timestamp("2025-12-31")
# df = df[a]

print(pd.Timestamp("2025-12-31"))

2025-12-31 00:00:00
